<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/BioEmu_Benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BioEmu Benchmarks**

This notebook runs the [BioEmu benchmarks](https://github.com/microsoft/bioemu-benchmarks) on `topology.pdb` + `samples.xtc` outputs from the [BioEmu Colab notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/BioEmu.ipynb).

## Available Benchmarks

| Benchmark | Description | Recommended Samples |
|-----------|-------------|--------------------|
| `multiconf_ood60` | Local conformational changes (out-of-distribution) | 4,000/protein |
| `multiconf_oodval` | Global conformational changes (validation) | 4,000/protein |
| `multiconf_domainmotion` | Global domain motions | 4,000/protein |
| `multiconf_crypticpocket` | Cryptic pocket backbone changes | 4,000/protein |
| `singleconf_localunfolding` | Local protein unfolding | 4,000/protein |
| `folding_free_energies` | Folding free energy prediction | 200-7,600/protein |
| `md_emulation` | MD distribution matching | 10,000/protein |

## Input Format

Upload your BioEmu samples organized as:
```
my_samples/
├── protein_1/
│   ├── topology.pdb
│   └── samples.xtc
├── protein_2/
│   ├── topology.pdb
│   └── samples.xtc
└── ...
```
Each benchmark requires samples for specific protein sequences. Use the **"Show benchmark specs"** cell to see which sequences are needed.

In [ ]:
#@title Install dependencies
import os
import sys

_is_bench_setup_file = '/content/.BIOEMU_BENCH_SETUP'

if not os.path.exists(_is_bench_setup_file):
    # Install bioemu-benchmarks and its dependencies
    os.system('pip install -q "bioemu-benchmarks @ git+https://github.com/microsoft/bioemu-benchmarks.git"')
    os.system(f'touch {_is_bench_setup_file}')
    print('Installation complete.')
else:
    print('Dependencies already installed.')

In [ ]:
#@title Configure sample directory and output
#@markdown - `samples_dir`: Path to directory containing your BioEmu samples (topology.pdb + samples.xtc pairs)
#@markdown - `output_dir`: Path where benchmark results will be saved
#@markdown - `filter_samples`: Filter out unphysical samples (chain breaks, clashes) before evaluation
#@markdown - `use_google_drive`: Mount Google Drive for input/output

samples_dir = "/content/my_samples" #@param {type:"string"}
output_dir = "/content/benchmark_results" #@param {type:"string"}
filter_samples = True #@param {type:"boolean"}
use_google_drive = False #@param {type:"boolean"}

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(output_dir, exist_ok=True)
print(f'Samples directory: {samples_dir}')
print(f'Output directory: {output_dir}')

In [ ]:
#@title Show benchmark specs (sequences & recommended sample sizes)
#@markdown Select a benchmark to see its required sequences and recommended sample counts.

benchmark_to_show = "multiconf_ood60" #@param ["multiconf_ood60", "multiconf_oodval", "multiconf_domainmotion", "multiconf_crypticpocket", "singleconf_localunfolding", "folding_free_energies", "md_emulation"]

from bioemu_benchmarks.benchmarks import Benchmark

bm = Benchmark(benchmark_to_show)
metadata = bm.metadata.copy()
metadata['recommended_samples'] = bm.default_samplesize
print(f'Benchmark: {benchmark_to_show}')
print(f'Number of test cases: {len(metadata)}')
print(f'Number of unique sequences: {metadata["sequence"].nunique()}')
print()
display(metadata)

In [ ]:
#@title Load and validate samples
#@markdown This cell discovers all topology.pdb + samples.xtc pairs in your samples directory
#@markdown and reports what was found.

import mdtraj
from bioemu_benchmarks.samples import find_samples_in_dir

sequence_samples = find_samples_in_dir(samples_dir)
print(f'Found {len(sequence_samples)} sample file(s):')
print()
for ss in sequence_samples:
    top = mdtraj.load_topology(ss.topology_file)
    seq = top.to_fasta()[0]
    traj = mdtraj.load(ss.trajectory_file, top=ss.topology_file)
    print(f'  {ss.trajectory_file}')
    print(f'    Sequence ({len(seq)} residues): {seq[:50]}{"..." if len(seq) > 50 else ""}')
    print(f'    Frames: {traj.n_frames}')
    print()

In [ ]:
#@title Helper: run a single benchmark

import json
import warnings
from pathlib import Path

from bioemu_benchmarks.benchmarks import Benchmark
from bioemu_benchmarks.evaluator_utils import evaluator_from_benchmark
from bioemu_benchmarks.samples import (
    IndexedSamples,
    NoSamples,
    filter_unphysical_samples,
    find_samples_in_dir,
)


def run_benchmark(benchmark_name, samples_dir, output_dir, filter_samples=True):
    """Run a single benchmark and return the results object."""
    benchmark = Benchmark(benchmark_name)
    results_dir = Path(output_dir) / benchmark_name
    results_dir.mkdir(parents=True, exist_ok=True)

    # Load samples
    sequence_samples = find_samples_in_dir(samples_dir)

    try:
        indexed_samples = IndexedSamples.from_benchmark(
            benchmark=benchmark, sequence_samples=sequence_samples
        )
    except NoSamples:
        print(f'No matching samples found for {benchmark_name}. '
              f'Use the "Show benchmark specs" cell to see required sequences.')
        return None

    matched_cases = list(indexed_samples.test_case_to_sequencesamples.keys())
    total_cases = len(benchmark.metadata)
    print(f'Matched {len(matched_cases)}/{total_cases} test cases')

    # Filter unphysical samples
    if filter_samples:
        print('Filtering unphysical samples...')
        indexed_samples, filter_stats = filter_unphysical_samples(indexed_samples)
        filter_stats_mean = {k: float(v.mean()) for k, v in filter_stats.items()}
        with open(results_dir / 'filter_statistics.json', 'w') as f:
            json.dump(filter_stats_mean, f, indent=2, sort_keys=True)
        avg_kept = sum(filter_stats_mean.values()) / len(filter_stats_mean) if filter_stats_mean else 0
        print(f'Average fraction of samples kept after filtering: {avg_kept:.2%}')

    # Run evaluation
    print(f'Running {benchmark_name} evaluation...')
    evaluator = evaluator_from_benchmark(benchmark=benchmark)
    results = evaluator(indexed_samples)

    # Save results
    print('Saving results...')
    results.save_results(results_dir)
    results.to_pickle(results_dir / 'results.pkl')

    # Plot
    print('Generating plots...')
    results.plot(results_dir)

    # Aggregate metrics
    aggregate = results.get_aggregate_metrics()
    with open(results_dir / 'aggregate_metrics.json', 'w') as f:
        json.dump(aggregate, f, indent=2, sort_keys=True)

    print(f'\nResults saved to: {results_dir}')
    print(f'\nAggregate metrics:')
    for k, v in aggregate.items():
        print(f'  {k}: {v:.4f}')

    return results

print('Helper loaded.')

---
## Run Individual Benchmarks
Run only the benchmarks for which you have matching samples. Each cell is independent.

In [ ]:
#@title Multiconf OOD60
#@markdown Evaluates local conformational changes on out-of-distribution proteins.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 22 test cases.

results_ood60 = run_benchmark('multiconf_ood60', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf OODVAL
#@markdown Evaluates global conformational changes on validation proteins.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 18 test cases.

results_oodval = run_benchmark('multiconf_oodval', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf Domain Motion
#@markdown Evaluates global protein domain motions.
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 27 test cases.

results_domainmotion = run_benchmark('multiconf_domainmotion', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Multiconf Cryptic Pocket
#@markdown Evaluates pocket backbone changes upon ligand binding (holo vs apo).
#@markdown Metrics: RMSD, TM-score, lDDT, contact distance, DSSP accuracy.
#@markdown Recommended: 4,000 samples per protein, 34 test cases.

results_crypticpocket = run_benchmark('multiconf_crypticpocket', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Singleconf Local Unfolding
#@markdown Evaluates local protein unfolding via fraction of native contacts.
#@markdown Metrics: FNC (fold/unfold regions).
#@markdown Recommended: 4,000 samples per protein, 20 test cases.

results_localunfolding = run_benchmark('singleconf_localunfolding', samples_dir, output_dir, filter_samples)

In [ ]:
#@title Folding Free Energies
#@markdown Predicts folding free energies (dG and ddG) from sample ensembles.
#@markdown Metrics: MAE and correlation for dG and ddG vs experiment.
#@markdown Recommended: 200-7,600 samples per protein, 364 test cases (~100+ unique sequences).

results_folding = run_benchmark('folding_free_energies', samples_dir, output_dir, filter_samples)

In [ ]:
#@title MD Emulation
#@markdown Compares sample distributions to MD reference on projected free energy surfaces.
#@markdown Metrics: Free energy MAE, RMSE, and coverage.
#@markdown Recommended: 10,000 samples per protein, 16 test cases.

results_md = run_benchmark('md_emulation', samples_dir, output_dir, filter_samples)

---
## Run All Benchmarks at Once
Alternatively, run all benchmarks in sequence. Only benchmarks with matching samples will produce results.

In [ ]:
#@title Run all benchmarks

all_benchmark_names = [
    'multiconf_ood60',
    'multiconf_oodval',
    'multiconf_domainmotion',
    'multiconf_crypticpocket',
    'singleconf_localunfolding',
    'folding_free_energies',
    'md_emulation',
]

all_results = {}
all_aggregate = {}

for bm_name in all_benchmark_names:
    print(f'\n{"=" * 60}')
    print(f'Running: {bm_name}')
    print(f'{"=" * 60}')
    result = run_benchmark(bm_name, samples_dir, output_dir, filter_samples)
    if result is not None:
        all_results[bm_name] = result
        all_aggregate[bm_name] = result.get_aggregate_metrics()

# Save combined aggregate metrics
combined_path = Path(output_dir) / 'benchmark_metrics.json'
with open(combined_path, 'w') as f:
    json.dump(all_aggregate, f, indent=2, sort_keys=True)

print(f'\n{"=" * 60}')
print(f'All benchmarks complete. Combined metrics saved to: {combined_path}')
print(f'{"=" * 60}')

In [ ]:
#@title Display generated plots
#@markdown Show plots from all completed benchmarks.

from IPython.display import display, Image
from pathlib import Path

results_path = Path(output_dir)
png_files = sorted(results_path.glob('**/*.png'))

if not png_files:
    print('No plots found. Run a benchmark first.')
else:
    for png in png_files:
        relative = png.relative_to(results_path)
        print(f'\n--- {relative} ---')
        display(Image(filename=str(png), width=600))